# Imports and Configuration

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib
import ipywidgets as widgets
from IPython.display import display, clear_output

matplotlib.rcParams["figure.dpi"] = 600

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


## Constants and Paths

Define shock data directory, default time deltas for upstream/downstream regions, and satellite-specific settings.

In [3]:
SHOCKS_DIR = Path("Data")
PARAMS_PATH = Path("Shocks/Shock Params.json")
DEFAULT_DTS = {"dt0_u": -5, "dt1_u": -2, "dt0_d": 2, "dt1_d": 5}

high_mag_density = {"ace", "dscovr"}
high_v_density = {"dscovr"}

# Shock Determination


## Data Loading Functions

Functions to load shock data from parquet files and manage the data cache.

In [4]:
def list_shock_dates():
    if not SHOCKS_DIR.exists():
        return []
    return sorted([p.name for p in SHOCKS_DIR.iterdir() if p.is_dir()])


def _ensure_datetime_index(df):
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for col in ["time", "Time", "datetime", "date", "timestamp", "Timestamp"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
            return df.set_index(col)
    df.index = pd.to_datetime(df.index)
    return df


def load_shock_folder(date):
    folder = SHOCKS_DIR / date
    dfs = {}
    for path in sorted(folder.glob("*.parquet")):
        sat = path.stem.lower()
        df = pd.read_parquet(path)
        df = _ensure_datetime_index(df)
        df = df.sort_index()
        # Remove duplicate indices, keeping first occurrence
        df = df[~df.index.duplicated(keep="first")]
        dfs[sat] = df
    return dfs


dfs_cache = {}


def get_dfs(date):
    if date not in dfs_cache:
        dfs_cache[date] = load_shock_folder(date)
    return dfs_cache[date]

## Parameter Management

Load, save, and manage shock parameters (t0, dt0_u, dt1_u, dt0_d, dt1_d) for each date and satellite combination.

In [5]:
def _seed_params_for_date(date, dfs):
    params = {}
    for sat, df in dfs.items():
        if df.empty:
            continue
        t0 = df.index[len(df) // 2]
        params[sat] = {"t0": t0, **DEFAULT_DTS}
    return params


def _parse_t0(value):
    if value is None:
        return None

    if isinstance(value, pd.Timestamp):
        return value

    if isinstance(value, (int, float, np.integer, np.floating)):
        # Legacy files may store epoch as ms (1e12) or ns (1e18).
        unit = "ns" if abs(float(value)) > 1e14 else "ms"
        return pd.to_datetime(int(value), unit=unit)

    if isinstance(value, str):
        try:
            return pd.Timestamp(value)
        except Exception:
            return None

    return None


def _normalize_sat_params(entry):
    if not isinstance(entry, dict):
        return None

    t0 = _parse_t0(entry.get("t0"))
    if t0 is None:
        return None

    return {
        "t0": t0,
        "dt0_u": int(entry.get("dt0_u", DEFAULT_DTS["dt0_u"])),
        "dt1_u": int(entry.get("dt1_u", DEFAULT_DTS["dt1_u"])),
        "dt0_d": int(entry.get("dt0_d", DEFAULT_DTS["dt0_d"])),
        "dt1_d": int(entry.get("dt1_d", DEFAULT_DTS["dt1_d"])),
    }


def _load_params():
    if not PARAMS_PATH.exists():
        return {}

    try:
        raw = json.loads(PARAMS_PATH.read_text())
    except Exception:
        return {}

    if not isinstance(raw, dict):
        return {}

    known_dates = set(list_shock_dates())
    out = {}

    for date_key, sats in raw.items():
        date = str(date_key)

        # Drop unknown / corrupted top-level keys to avoid cross-pollination.
        if date not in known_dates:
            continue

        if not isinstance(sats, dict):
            continue

        out[date] = {}
        for sat, entry in sats.items():
            normalized = _normalize_sat_params(entry)
            if normalized is not None:
                out[date][str(sat)] = normalized

    return out


def _serialize_params(data):
    payload = {}

    for date in sorted(list_shock_dates()):
        sats = data.get(date, {})
        if not isinstance(sats, dict):
            continue

        sat_payload = {}
        for sat, entry in sats.items():
            normalized = _normalize_sat_params(entry)
            if normalized is None:
                continue

            sat_payload[str(sat)] = {
                "t0": pd.Timestamp(normalized["t0"]).isoformat(),
                "dt0_u": int(normalized["dt0_u"]),
                "dt1_u": int(normalized["dt1_u"]),
                "dt0_d": int(normalized["dt0_d"]),
                "dt1_d": int(normalized["dt1_d"]),
            }

        payload[date] = sat_payload

    return payload


def _save_params(data):
    payload = _serialize_params(data)
    tmp_path = PARAMS_PATH.with_suffix(".json.tmp")
    tmp_path.write_text(json.dumps(payload, indent=2, sort_keys=True))
    tmp_path.replace(PARAMS_PATH)


def _ensure_date_initialized(date):
    """Seed Shock Params.json once for a truly new date."""
    if date in data:
        return data[date]

    seeded = _seed_params_for_date(date, get_dfs(date))
    data[date] = seeded
    if seeded:
        _save_params(data)
    return data[date]


def _get_date_params(date):
    _ensure_date_initialized(date)
    return data.get(date, {})


data = _load_params()

# Provisional (in-memory only) params for satellites not yet confirmed.
# These render in the plot but are NOT saved to JSON until the user moves a slider.
_provisional_params = {}


def _ensure_params_for(date, sat):
    """Ensure in-memory params exist for (date, sat). Does NOT save to JSON."""
    _ensure_date_initialized(date)
    if date in data and sat in data[date] and data[date][sat] is not None:
        return
    if date in _provisional_params and sat in _provisional_params.get(date, {}):
        return
    df = get_dfs(date)[sat]
    t0 = df.index[len(df) // 2]
    _provisional_params.setdefault(date, {})[sat] = {"t0": t0, **DEFAULT_DTS}


def _get_params_for(date, sat):
    """Get params from confirmed data or provisional. Returns None if neither."""
    if date in data and sat in data[date] and data[date][sat] is not None:
        return data[date][sat]
    if date in _provisional_params and sat in _provisional_params.get(date, {}):
        return _provisional_params[date][sat]
    return None


def _is_provisional(date, sat):
    """Check whether (date, sat) params are provisional (not saved to JSON)."""
    return date in _provisional_params and sat in _provisional_params.get(date, {})


def _promote_provisional(date, sat):
    """Move a provisional satellite into confirmed data and save."""
    if not _is_provisional(date, sat):
        return
    entry = _provisional_params[date].pop(sat)
    if not _provisional_params[date]:
        del _provisional_params[date]
    data.setdefault(date, {})[sat] = entry
    _save_params(data)


def _invalidate_sat(date, sat):
    """Remove satellite from confirmed data and provisional. Saves JSON."""
    changed = False
    if date in data and sat in data[date]:
        del data[date][sat]
        changed = True
    if date in _provisional_params and sat in _provisional_params.get(date, {}):
        del _provisional_params[date][sat]
        if not _provisional_params[date]:
            del _provisional_params[date]
    if changed:
        _save_params(data)

## Plotting Function

Main function to visualize shock data with magnetic field, velocity, and density plots. Includes support for multiple coordinate systems and satellite-specific configurations.

In [6]:
def _nearest_index(df, t0):
    # Handle unique index case
    if df.index.is_unique:
        return df.index[df.index.get_indexer([t0], method="nearest")[0]]
    else:
        # For non-unique index, find the closest value manually
        return min(df.index, key=lambda x: abs((x - t0).total_seconds()))


def _plot_window(df, center_t, mode="medium"):
    if mode == "coarse":
        return df

    if mode == "fine":
        delta = pd.Timedelta(seconds=90)
    else:
        delta = pd.Timedelta(minutes=30)

    start = pd.Timestamp(center_t) - delta
    end = pd.Timestamp(center_t) + delta
    df_window = df[start:end]
    return df_window if len(df_window) > 0 else df


def _draw_reference_lines(axis, t0, dt0_u, dt1_u, dt0_d, dt1_d):
    axis.axvline(
        x=pd.Timestamp(t0), color="black", linewidth=2, linestyle="--", label="Shock: " + t0.strftime("%Y-%m-%d %H:%M:%S")
    )
    for key, dt, color in [
        ("dt0_u", dt0_u, "tab:red"),
        ("dt1_u", dt1_u, "tab:orange"),
        ("dt0_d", dt0_d, "tab:blue"),
        ("dt1_d", dt1_d, "tab:purple"),
    ]:
        axis.axvline(
            x=pd.Timestamp(t0) + pd.Timedelta(minutes=dt),
            color=color,
            linewidth=1.2,
            linestyle=":",
            alpha=0.9,
            label="_nolegend_",
        )


def _select_density_column(df_plot):
    for col in ["N_p", "n_p", "Proton_Np_moment", "Np", "N", "proton_density"]:
        if col in df_plot.columns:
            return col
    return None


def plot(
    date, sat, t0=None, dt0_u=None, dt1_u=None, dt0_d=None, dt1_d=None, mode="medium"
):
    dfs = get_dfs(date)
    if sat not in dfs:
        print(f"No data for {sat} in {date}")
        return

    df = dfs[sat]
    _ensure_params_for(date, sat)
    dat = _get_params_for(date, sat)

    t00 = pd.Timestamp(dat["t0"])

    # Ensure timezone consistency
    if len(df.index) > 0:
        if df.index[0].tz is not None and t00.tz is None:
            t00 = t00.tz_localize("UTC")
        elif df.index[0].tz is None and t00.tz is not None:
            t00 = t00.tz_localize(None)

    if t0 is None:
        t0 = t00
    else:
        # Ensure t0 from slider is also timezone-consistent
        t0 = pd.Timestamp(t0)
        if df.index[0].tz is not None and t0.tz is None:
            t0 = t0.tz_localize("UTC")
        elif df.index[0].tz is None and t0.tz is not None:
            t0 = t0.tz_localize(None)
        dat["t0"] = t0

    if dt0_u is None:
        dt0_u = dat["dt0_u"]
    else:
        dat["dt0_u"] = dt0_u
    if dt1_u is None:
        dt1_u = dat["dt1_u"]
    else:
        dat["dt1_u"] = dt1_u
    if dt0_d is None:
        dt0_d = dat["dt0_d"]
    else:
        dat["dt0_d"] = dt0_d
    if dt1_d is None:
        dt1_d = dat["dt1_d"]
    else:
        dat["dt1_d"] = dt1_d

    # Select time range based on the current selected t0
    df_plot = _plot_window(df, t0, mode=mode)

    fig = plt.figure(figsize=(12, 10))
    plt.rcParams["lines.markersize"] = 2.5
    ax = plt.subplot2grid((3, 1), (0, 0), sharex=None)

    marker = "-o" if sat not in high_mag_density else "-"

    ax.set_xlim(df_plot.index[0], df_plot.index[-1])

    def plot_col(a, col, label, color=None):
        if col in df_plot.columns:
            if color is None:
                a.plot(
                    df_plot.dropna(subset=[col]).index,
                    df_plot[col].dropna(),
                    marker,
                    label=label,
                )
            else:
                a.plot(
                    df_plot.dropna(subset=[col]).index,
                    df_plot[col].dropna(),
                    marker,
                    label=label,
                    color=color,
                )

    # Magnetic field - magnitude always
    if "B" in df_plot.columns:
        plot_col(ax, "B", "B")
    elif all(c in df_plot.columns for c in ["B_X_GSE", "B_Y_GSE", "B_Z_GSE"]):
        bmag = np.sqrt(
            df_plot["B_X_GSE"] ** 2 + df_plot["B_Y_GSE"] ** 2 + df_plot["B_Z_GSE"] ** 2
        )
        ax.plot(df_plot.index, bmag, marker, label="B")
    elif all(c in df_plot.columns for c in ["B_X_HGE", "B_Y_HGE", "B_Z_HGE"]):
        bmag = np.sqrt(
            df_plot["B_X_HGE"] ** 2 + df_plot["B_Y_HGE"] ** 2 + df_plot["B_Z_HGE"] ** 2
        )
        ax.plot(df_plot.index, bmag, marker, label="B")
    elif all(c in df_plot.columns for c in ["B_x", "B_y", "B_z"]):
        bmag = np.sqrt(df_plot["B_x"] ** 2 + df_plot["B_y"] ** 2 + df_plot["B_z"] ** 2)
        ax.plot(df_plot.index, bmag, marker, label="B")

    # Plot B components only for non-STEREO (GSE coordinates)
    if sat != "stereo":
        if "B_X_GSE" in df_plot.columns:
            plot_col(ax, "B_X_GSE", "B_x")
            plot_col(ax, "B_Y_GSE", "B_y")
            plot_col(ax, "B_Z_GSE", "B_z")
        else:
            plot_col(ax, "B_x", "B_x")
            plot_col(ax, "B_y", "B_y")
            plot_col(ax, "B_z", "B_z")

    _draw_reference_lines(ax, t0, dt0_u, dt1_u, dt0_d, dt1_d)
    ax.legend(loc="upper left")
    ax.set_ylabel("Magnetic Field (nT)")

    marker = "-o" if sat not in high_v_density else "-"
    plt.rcParams["lines.markersize"] = 5

    ax1 = plt.subplot2grid((3, 1), (1, 0), sharex=ax)
    ax1.set_xlim(df_plot.index[0], df_plot.index[-1])

    # Determine if we're in GSE coordinates
    is_gse = "V_X_GSE" in df_plot.columns or "v_x" in df_plot.columns

    # Velocity plotting with multiple naming convention support
    if "V" in df_plot.columns:
        plot_col(ax1, "V", "V")
    elif "v" in df_plot.columns:
        plot_col(ax1, "v", "V")
    elif "Proton_V_moment" in df_plot.columns:
        plot_col(ax1, "Proton_V_moment", "V")
    elif all(c in df_plot.columns for c in ["V_X_GSE", "V_Y_GSE", "V_Z_GSE"]):
        vmag = np.sqrt(
            df_plot["V_X_GSE"] ** 2 + df_plot["V_Y_GSE"] ** 2 + df_plot["V_Z_GSE"] ** 2
        )
        ax1.plot(df_plot.index, vmag, marker, label="V")
    elif all(c in df_plot.columns for c in ["V_X_HGE", "V_Y_HGE", "V_Z_HGE"]):
        vmag = np.sqrt(
            df_plot["V_X_HGE"] ** 2 + df_plot["V_Y_HGE"] ** 2 + df_plot["V_Z_HGE"] ** 2
        )
        ax1.plot(df_plot.index, vmag, marker, label="V")
    elif all(c in df_plot.columns for c in ["v_x", "v_y", "v_z"]):
        vmag = np.sqrt(df_plot["v_x"] ** 2 + df_plot["v_y"] ** 2 + df_plot["v_z"] ** 2)
        ax1.plot(df_plot.index, vmag, marker, label="V")
    elif all(
        c in df_plot.columns
        for c in ["Proton_VX_moment", "Proton_VY_moment", "Proton_VZ_moment"]
    ):
        vmag = np.sqrt(
            df_plot["Proton_VX_moment"] ** 2
            + df_plot["Proton_VY_moment"] ** 2
            + df_plot["Proton_VZ_moment"] ** 2
        )
        ax1.plot(
            df_plot.dropna(subset=["Proton_VX_moment"]).index,
            vmag[df_plot["Proton_VX_moment"].notna()],
            marker,
            label="V",
        )

    # Plot V_x (or -V_x if GSE) on the first axis
    if "V_X_GSE" in df_plot.columns:
        vx_data = -df_plot["V_X_GSE"] if is_gse else df_plot["V_X_GSE"]
        ax1.plot(
            df_plot.dropna(subset=["V_X_GSE"]).index,
            vx_data.dropna(),
            marker,
            label="-V_x" if is_gse else "V_x",
        )
    elif "v_x" in df_plot.columns:
        vx_data = -df_plot["v_x"] if is_gse else df_plot["v_x"]
        ax1.plot(
            df_plot.dropna(subset=["v_x"]).index,
            vx_data.dropna(),
            marker,
            label="-V_x" if is_gse else "V_x",
        )

    # Create second y-axis for density
    density_col = _select_density_column(df_plot)
    ax1_density = None
    if density_col is not None:
        ax1_density = ax1.twinx()
        ax1_density.plot(
            df_plot.dropna(subset=[density_col]).index,
            df_plot[density_col].dropna(),
            "-",
            color="tab:purple",
            linewidth=1.3,
            alpha=0.85,
            label="N_p",
        )
        ax1_density.set_ylabel("Density (cm^-3)")

    # Create third y-axis for V_y and V_z
    ax1_yz = ax1.twinx()
    ax1_yz.spines["right"].set_position(("outward", 60))

    if "V_Y_GSE" in df_plot.columns and "V_Z_GSE" in df_plot.columns:
        ax1_yz.plot(
            df_plot.dropna(subset=["V_Y_GSE"]).index,
            df_plot["V_Y_GSE"].dropna(),
            marker,
            label="V_y",
            alpha=0.7,
            color="green",
        )
        ax1_yz.plot(
            df_plot.dropna(subset=["V_Z_GSE"]).index,
            df_plot["V_Z_GSE"].dropna(),
            marker,
            label="V_z",
            alpha=0.7,
            color="red",
        )
    elif "v_y" in df_plot.columns and "v_z" in df_plot.columns:
        ax1_yz.plot(
            df_plot.dropna(subset=["v_y"]).index,
            df_plot["v_y"].dropna(),
            marker,
            label="V_y",
            alpha=0.7,
            color="green",
        )
        ax1_yz.plot(
            df_plot.dropna(subset=["v_z"]).index,
            df_plot["v_z"].dropna(),
            marker,
            label="V_z",
            alpha=0.7,
            color="red",
        )
    elif "V_Y_HGE" in df_plot.columns and "V_Z_HGE" in df_plot.columns:
        ax1_yz.plot(
            df_plot.dropna(subset=["V_Y_HGE"]).index,
            df_plot["V_Y_HGE"].dropna(),
            marker,
            label="V_y",
            alpha=0.7,
            color="green",
        )
        ax1_yz.plot(
            df_plot.dropna(subset=["V_Z_HGE"]).index,
            df_plot["V_Z_HGE"].dropna(),
            marker,
            label="V_z",
            alpha=0.7,
            color="red",
        )

    ax1_yz.set_ylabel("V_y, V_z (km/s)")

    _draw_reference_lines(ax1, t0, dt0_u, dt1_u, dt0_d, dt1_d)

    # Combine all legends
    handles, labels = ax1.get_legend_handles_labels()
    if ax1_density is not None:
        h2, l2 = ax1_density.get_legend_handles_labels()
        handles.extend(h2)
        labels.extend(l2)
    h3, l3 = ax1_yz.get_legend_handles_labels()
    handles.extend(h3)
    labels.extend(l3)
    ax1.legend(handles, labels, loc="upper left")
    ax1.set_ylabel("V, V_x (km/s)")

    fig.tight_layout()
    plt.show()
    if not _is_provisional(date, sat):
        _save_params(data)

## Satellite Locations

Location-only companion plot for the interactive explorer.


In [ ]:
R_E = 6371.0

_SATS_ALREADY_RE = {"wind"}  # Wind parquets store positions in R_E


def _get_sat_params(params, sat_name):
    sat = str(sat_name).lower()
    candidates = [sat]
    if sat == "mms1":
        candidates.append("mms_1")
    if sat == "mms_1":
        candidates.append("mms1")
    for key in candidates:
        if key in params and params[key] is not None:
            return params[key]
    return None


def _pick_position_columns(df):
    for cols in [
        ("X_GSE", "Y_GSE", "Z_GSE"),
        ("x_gse", "y_gse", "z_gse"),
        ("X", "Y", "Z"),
        ("x", "y", "z"),
    ]:
        if all(c in df.columns for c in cols):
            return cols
    return None


def _coords_already_re(x, y, z, sat_name=None):
    sat = str(sat_name).lower() if sat_name is not None else ""
    if sat in _SATS_ALREADY_RE:
        return True
    max_abs = np.nanmax(np.abs([x, y, z]))
    return max_abs < 1000


def get_sat_position_re(df, t0, sat_name=None):
    pos_cols = _pick_position_columns(df)
    if pos_cols is None:
        return None

    pos_df = df.loc[:, list(pos_cols)].copy()
    pos_df = pos_df.apply(pd.to_numeric, errors="coerce")
    pos_df = pos_df.sort_index()
    pos_df = pos_df[~pos_df.index.duplicated(keep="first")]

    t0_ts = pd.Timestamp(t0)
    if pos_df.index.tz is not None and t0_ts.tz is None:
        t0_ts = t0_ts.tz_localize("UTC")
    elif pos_df.index.tz is None and t0_ts.tz is not None:
        t0_ts = t0_ts.tz_localize(None)

    idx = pos_df.index.get_indexer([t0_ts], method="nearest")[0]
    row = pos_df.iloc[idx]
    x, y, z = row[pos_cols[0]], row[pos_cols[1]], row[pos_cols[2]]

    if any(np.isnan(v) for v in (x, y, z)):
        window = pos_df[
            t0_ts - pd.Timedelta(minutes=60) : t0_ts + pd.Timedelta(minutes=60)
        ]
        valid = window.dropna(subset=list(pos_cols))
        if valid.empty:
            valid = pos_df.dropna(subset=list(pos_cols))
            if valid.empty:
                return None
        nearest_idx = valid.index.get_indexer([t0_ts], method="nearest")[0]
        row = valid.iloc[nearest_idx]
        x, y, z = row[pos_cols[0]], row[pos_cols[1]], row[pos_cols[2]]

    if not _coords_already_re(x, y, z, sat_name=sat_name):
        x, y, z = x / R_E, y / R_E, z / R_E

    return {"x": float(x), "y": float(y), "z": float(z)}


def plot_satellite_locations(date, selected_sat=None):
    dfs_local = get_dfs(date)
    params = _get_date_params(date)

    fig, (ax_xy, ax_xz) = plt.subplots(
        ncols=2, figsize=(8, 5), gridspec_kw={"wspace": 0.4}
    )

    for axi in (ax_xy, ax_xz):
        axi.tick_params(
            axis="both",
            which="both",
            top=True,
            bottom=True,
            left=True,
            right=True,
            direction="in",
        )
        for spine in ["top", "bottom", "left", "right"]:
            axi.spines[spine].set_visible(True)
        axi.axhline(y=0, color="black", linewidth=0.5)
        axi.axvline(x=0, color="black", linewidth=0.5)

    ax_xy.set_xlim(-150, 400)
    ax_xy.set_ylim(-200, 300)
    ax_xz.set_xlim(-150, 400)
    ax_xz.set_ylim(-200, 300)

    excluded_sats = {"stereo", "themis_c"}
    sat_list = []
    sat_positions = {}

    for sat0 in sorted(dfs_local.keys()):
        sat_key = str(sat0).lower()
        if sat_key in excluded_sats or sat_key.startswith("stereo"):
            continue
        p = _get_sat_params(params, sat_key)
        if p is None:
            continue
        pos = get_sat_position_re(dfs_local[sat0], p["t0"], sat_name=sat0)
        if pos is None:
            continue
        sat_list.append(sat0)
        sat_positions[sat0] = pos

    sat_colors = {sat0: f"C{i}" for i, sat0 in enumerate(sat_list)}
    selected_key = str(selected_sat).lower() if selected_sat is not None else None

    for sat0 in sat_list:
        pos = sat_positions[sat0]
        color = sat_colors[sat0]
        is_selected = str(sat0).lower() == selected_key
        marker = "*" if is_selected else "o"
        size = 90 if is_selected else 35
        edge = "black" if is_selected else "none"
        ax_xy.scatter(pos["x"], pos["y"], color=color, s=size, marker=marker, edgecolors=edge, linewidths=0.8, zorder=10)
        ax_xy.text(pos["x"], pos["y"], f" {sat0}", fontsize=8, va="center")
        ax_xz.scatter(pos["x"], pos["z"], color=color, s=size, marker=marker, edgecolors=edge, linewidths=0.8, zorder=10)
        ax_xz.text(pos["x"], pos["z"], f" {sat0}", fontsize=8, va="center")

    ax_xy.set_aspect("equal")
    ax_xz.set_aspect("equal")
    ax_xy.set_xlabel("-X (GSE), $R_E$")
    ax_xy.set_ylabel("-Y (GSE), $R_E$")
    ax_xz.set_xlabel("-X (GSE), $R_E$")
    ax_xz.set_ylabel("Z (GSE), $R_E$")
    ax_xy.invert_xaxis()
    ax_xy.invert_yaxis()
    ax_xz.invert_xaxis()
    fig.suptitle(f"Satellite Locations - {date}", fontsize=14)
    fig.subplots_adjust(top=0.9)
    plt.show()


## Interactive Controls

Create widgets for interactive parameter tuning: date selection, satellite selection, shock time (t0), and upstream/downstream time deltas.

In [7]:
dates = list_shock_dates()
if not dates:
    print("No shock folders found in Data")

date_dropdown = widgets.Dropdown(
    options=dates, description="Date", layout=widgets.Layout(width="50%")
)
sat_dropdown = widgets.Dropdown(
    options=[pd.Timestamp("1970-01-01")],
    description="Sat",
    layout=widgets.Layout(width="50%"),
)

t0_slider = widgets.SelectionSlider(
    options=[pd.Timestamp("1970-01-01")],
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    layout=widgets.Layout(width="75%"),
)

dt0_u_slider = widgets.IntSlider(
    value=DEFAULT_DTS["dt0_u"],
    max=-1,
    min=-60,
    step=1,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
    description="dt0_u (min)",
    layout=widgets.Layout(width="75%"),
)
dt1_u_slider = widgets.IntSlider(
    value=DEFAULT_DTS["dt1_u"],
    max=-1,
    min=-60,
    step=1,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
    description="dt1_u (min)",
    layout=widgets.Layout(width="75%"),
)
dt0_d_slider = widgets.IntSlider(
    value=DEFAULT_DTS["dt0_d"],
    min=1,
    max=60,
    step=1,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
    description="dt0_d (min)",
    layout=widgets.Layout(width="75%"),
)
dt1_d_slider = widgets.IntSlider(
    value=DEFAULT_DTS["dt1_d"],
    min=1,
    max=60,
    step=1,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
    description="dt1_d (min)",
    layout=widgets.Layout(width="75%"),
)

mode_radio = widgets.RadioButtons(
    options=[("Coarse", "coarse"), ("Medium", "medium"), ("Fine", "fine")],
    value="medium",
    description="Mode",
    layout=widgets.Layout(width="50%"),
)

invalidate_btn = widgets.Button(
    description="Invalidate",
    button_style="danger",
    tooltip="Remove this satellite from Shock Params.json",
    layout=widgets.Layout(width="auto"),
)

status_label = widgets.Label(value="")

_control_update_in_progress = False
_last_sat = None


def _update_status_label():
    date = date_dropdown.value
    sat = sat_dropdown.value
    if not date or not sat:
        status_label.value = ""
        return
    if _is_provisional(date, sat):
        status_label.value = "\u26a0 Unsaved (move a slider to confirm)"
    elif date in data and sat in data.get(date, {}):
        status_label.value = "\u2705 Saved"
    else:
        status_label.value = ""


def _render(*_):
    if _control_update_in_progress:
        return
    if not date_dropdown.value or not sat_dropdown.value:
        return
    with out:
        clear_output(wait=True)
        plot(
            date=date_dropdown.value,
            sat=sat_dropdown.value,
            t0=t0_slider.value,
            dt0_u=dt0_u_slider.value,
            dt1_u=dt1_u_slider.value,
            dt0_d=dt0_d_slider.value,
            dt1_d=dt1_d_slider.value,
            mode=mode_radio.value,
        )
        plot_satellite_locations(date=date_dropdown.value, selected_sat=sat_dropdown.value)
    _update_status_label()


def _on_slider_change(*_):
    """Promote provisional params on any slider interaction, then render."""
    if _control_update_in_progress:
        return
    date = date_dropdown.value
    sat = sat_dropdown.value
    if date and sat and _is_provisional(date, sat):
        _promote_provisional(date, sat)
    _render()


def _on_invalidate(*_):
    date = date_dropdown.value
    sat = sat_dropdown.value
    if not date or not sat:
        return
    _invalidate_sat(date, sat)
    # Re-create provisional entry so the plot still renders
    _ensure_params_for(date, sat)
    _update_status_label()
    _render()


def _refresh_sats(*_):
    if not date_dropdown.value:
        sat_dropdown.options = []
        return
    sats = sorted(get_dfs(date_dropdown.value).keys())
    sat_dropdown.options = sats
    if sats and sat_dropdown.value not in sats:
        sat_dropdown.value = sats[0]
    else:
        _refresh_controls()


def _nearest_from_index(idx, t0):
    if idx.is_unique:
        return idx[idx.get_indexer([t0], method="nearest")[0]]
    return min(idx, key=lambda x: abs((x - t0).total_seconds()))


def _refresh_controls(*_):
    global _control_update_in_progress, _last_sat
    if _control_update_in_progress:
        return
    if not date_dropdown.value or not sat_dropdown.value:
        return

    _control_update_in_progress = True
    try:
        date = date_dropdown.value
        sat = sat_dropdown.value
        dfs = get_dfs(date)
        df = dfs[sat]
        _ensure_params_for(date, sat)
        dat = _get_params_for(date, sat)

        sat_changed = sat != _last_sat
        _last_sat = sat

        t0 = pd.Timestamp(dat["t0"])

        if len(df.index) > 0:
            idx_start = df.index[0]
            idx_end = df.index[-1]
            if idx_start.tz is not None and t0.tz is None:
                t0 = t0.tz_localize("UTC")
            elif idx_start.tz is None and t0.tz is not None:
                t0 = t0.tz_localize(None)

            # Only carry over the current slider value when the satellite
            # has NOT changed (e.g. mode switch).  When the satellite
            # changes we must use the value stored in JSON to avoid
            # cross-pollination from the previous satellite.
            if not sat_changed and t0_slider.value is not None:
                current_t0 = pd.Timestamp(t0_slider.value)
                if idx_start.tz is not None and current_t0.tz is None:
                    current_t0 = current_t0.tz_localize("UTC")
                elif idx_start.tz is None and current_t0.tz is not None:
                    current_t0 = current_t0.tz_localize(None)
                if idx_start <= current_t0 <= idx_end:
                    t0 = current_t0

        if t0 < df.index[0] or t0 > df.index[-1]:
            t0 = df.index[len(df) // 2]
            dat["t0"] = t0

        mode = mode_radio.value
        if mode == "coarse":
            t0_options = df.index
        elif mode == "fine":
            t_start = t0 - pd.Timedelta(seconds=90)
            t_end = t0 + pd.Timedelta(seconds=90)
            t0_options = df[t_start:t_end].index
            if len(t0_options) == 0:
                t0_options = df.index
        else:
            t_start = t0 - pd.Timedelta(minutes=30)
            t_end = t0 + pd.Timedelta(minutes=30)
            t0_options = df[t_start:t_end].index
            if len(t0_options) == 0:
                t0_options = df.index

        if tuple(t0_slider.options) != tuple(t0_options):
            t0_slider.options = t0_options

        nearest_t0 = _nearest_from_index(t0_options, t0)
        if t0_slider.value != nearest_t0:
            t0_slider.value = nearest_t0

        new_dt0_u = dat.get("dt0_u", DEFAULT_DTS["dt0_u"])
        new_dt1_u = dat.get("dt1_u", DEFAULT_DTS["dt1_u"])
        new_dt0_d = dat.get("dt0_d", DEFAULT_DTS["dt0_d"])
        new_dt1_d = dat.get("dt1_d", DEFAULT_DTS["dt1_d"])

        if dt0_u_slider.value != new_dt0_u:
            dt0_u_slider.value = new_dt0_u
        if dt1_u_slider.value != new_dt1_u:
            dt1_u_slider.value = new_dt1_u
        if dt0_d_slider.value != new_dt0_d:
            dt0_d_slider.value = new_dt0_d
        if dt1_d_slider.value != new_dt1_d:
            dt1_d_slider.value = new_dt1_d
    finally:
        _control_update_in_progress = False

    _update_status_label()
    _render()


out = widgets.Output()

# Refresh chain observers
# Date only refreshes satellites; controls refresh is driven from sat selection.
date_dropdown.observe(_refresh_sats, names="value")
sat_dropdown.observe(_refresh_controls, names="value")
mode_radio.observe(_refresh_controls, names="value")

# Slider observers — promote provisional on any slider interaction
t0_slider.observe(_on_slider_change, names="value")
dt0_u_slider.observe(_on_slider_change, names="value")
dt1_u_slider.observe(_on_slider_change, names="value")
dt0_d_slider.observe(_on_slider_change, names="value")
dt1_d_slider.observe(_on_slider_change, names="value")

invalidate_btn.on_click(_on_invalidate)

_refresh_sats()
_refresh_controls()

controls = widgets.VBox(
    [
        widgets.HBox([date_dropdown, sat_dropdown]),
        mode_radio,
        t0_slider,
        dt0_u_slider,
        dt1_u_slider,
        dt0_d_slider,
        dt1_d_slider,
        widgets.HBox([invalidate_btn, status_label]),
    ]
)

## Display

Show the interactive controls and output area for plots.

In [8]:
display(controls, out)

Output()